# FAQ Bot for a Single Website

## 1 — Project Overview

This notebook builds a **FAQ chatbot** that can answer questions about any website's content.

**End-to-end pipeline:**

```
Target URL
  → Crawl & extract text from pages
    → Split text into chunks
      → Build TF-IDF vector index
        → Retrieve relevant chunks for a question
          → Generate grounded answer (LLM or rule-based)
```

**Engineering patterns taught:**

| Pattern | Description |
|---------|-------------|
| Web scraping | Extract clean text from HTML using stdlib only |
| Text chunking | Split long documents into retrieval-friendly chunks with overlap |
| TF-IDF retrieval | Sparse vector search — no GPU, no embeddings API needed |
| RAG pipeline | Retrieve-then-generate with source grounding |
| Graceful fallback | Full pipeline works without LLM using extractive answers |

**Stack:** Python standard library + scikit-learn (TF-IDF). Ollama optional for generative answers.

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Crawl a website** — extract text from multiple pages within a single domain using only the standard library
2. **Chunk text for retrieval** — split long documents into overlapping chunks that preserve context
3. **Build a TF-IDF retrieval index** — create a sparse vector search index with scikit-learn
4. **Implement a RAG pipeline** — retrieve relevant chunks and generate grounded answers
5. **Evaluate retrieval quality** — measure whether the correct chunks are retrieved for known questions
6. **Compare extractive vs. generative answers** — understand the quality-reliability tradeoff

## 3 — Problem Statement

**Problem:** You have a website with dozens of pages (documentation, FAQ, product info). Users want to ask natural language questions and get accurate answers — without reading every page.

**Why not just use Ctrl+F?**
- Users don't know the exact keywords
- Answers may span multiple pages
- Users want synthesized answers, not raw page hits

**Constraints:**
- Single-domain crawl only (respect `robots.txt` in production)
- No external embedding APIs — must work offline
- Must handle the LLM being unavailable (extractive fallback)
- Must show which pages/chunks the answer came from (grounding)

## 4 — Why This Project Matters

**Website FAQ bots are a mainstream RAG use case:**
- Every SaaS company wants a support bot trained on their docs
- Internal knowledge bases (Confluence, Notion) use the same pattern
- Customer-facing FAQ bots reduce support ticket volume by 20-40%

**The website-to-RAG pipeline is the simplest production RAG pattern:**

| Component | This notebook | Production upgrade |
|-----------|---------------|-------------------|
| Ingestion | urllib + HTMLParser | Scrapy, Playwright, Firecrawl |
| Chunking | Fixed-size + overlap | Semantic chunking, Markdown-aware |
| Embeddings | TF-IDF (sparse) | Sentence Transformers, BGE-M3 |
| Vector store | In-memory numpy | Qdrant, Chroma, Weaviate |
| Generation | Ollama / extractive | GPT-4o, Claude, Qwen3 |

Master this pattern and every other RAG system is a variation of it.

## 5 — Website / Data Overview

We use **Python's official FAQ pages** (`docs.python.org/3/faq/`) as our target:

| Property | Value |
|----------|-------|
| URL | https://docs.python.org/3/faq/ |
| Pages | ~7 FAQ category pages |
| Content type | Technical Q&A about Python |
| License | PSF License (open) |

**Why this website?**
- Small enough to crawl in seconds
- Rich Q&A content (perfect for evaluation)
- Stable — won't change between runs
- No authentication or JS rendering needed

**Fallback:** If the website is unreachable, we include embedded sample content so the notebook runs offline.

## 6 — Environment Setup

We check two optional capabilities:
1. **Network access** — can we crawl the live website?
2. **Ollama** — can we generate answers with an LLM?

The notebook works fully without either (using embedded data + extractive answers).

In [ ]:
import urllib.request
import urllib.error
import json
from html.parser import HTMLParser


def is_ollama_available(host: str = "http://localhost:11434") -> bool:
    """Check if Ollama server is running locally."""
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False


def is_network_available() -> bool:
    """Check if we can reach the target website."""
    try:
        with urllib.request.urlopen("https://docs.python.org/3/faq/", timeout=5) as resp:
            return resp.status == 200
    except Exception:
        return False


OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"
USE_OLLAMA = is_ollama_available(OLLAMA_HOST)
HAS_NETWORK = is_network_available()

print(f"Network access:  {HAS_NETWORK}")
print(f"Ollama available: {USE_OLLAMA}")
if not HAS_NETWORK:
    print("  -> Will use embedded sample content")
if not USE_OLLAMA:
    print("  -> Will use extractive (non-LLM) answer generation")


## 7 — Imports

In [ ]:
import re
import time
from typing import Any, Dict, List, Tuple
from urllib.parse import urljoin, urlparse

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All imports loaded.")


## 8 — Data Loading (Website Extraction)

### Web scraping approach

We use Python's **stdlib only** — no Scrapy, no BeautifulSoup, no Playwright:

1. `urllib.request` to fetch HTML
2. A custom `HTMLParser` subclass to extract visible text (skip scripts, styles, nav)
3. Simple link extraction to discover pages within the same domain

**Key design decisions:**
- Crawl only pages under the same path prefix (stay within `/faq/`)
- Respect a `max_pages` limit to avoid hammering the server
- Add a 0.5s delay between requests (polite crawling)

In [ ]:
class TextExtractor(HTMLParser):
    """Extract visible text from HTML, skipping scripts/styles/nav."""

    SKIP_TAGS = {"script", "style", "nav", "header", "footer", "noscript", "svg"}

    def __init__(self):
        super().__init__()
        self.text_parts: List[str] = []
        self.links: List[str] = []
        self._skip_depth = 0

    def handle_starttag(self, tag, attrs):
        if tag in self.SKIP_TAGS:
            self._skip_depth += 1
        if tag == "a":
            for name, value in attrs:
                if name == "href" and value:
                    self.links.append(value)

    def handle_endtag(self, tag):
        if tag in self.SKIP_TAGS and self._skip_depth > 0:
            self._skip_depth -= 1

    def handle_data(self, data):
        if self._skip_depth == 0:
            cleaned = data.strip()
            if cleaned:
                self.text_parts.append(cleaned)

    def get_text(self) -> str:
        return "\n".join(self.text_parts)

    def get_links(self) -> List[str]:
        return self.links


def fetch_page(url: str) -> Tuple[str, List[str]]:
    """Fetch a URL and return (text, links)."""
    req = urllib.request.Request(url, headers={"User-Agent": "PythonFAQBot/1.0"})
    with urllib.request.urlopen(req, timeout=10) as resp:
        html = resp.read().decode("utf-8", errors="replace")
    parser = TextExtractor()
    parser.feed(html)
    return parser.get_text(), parser.get_links()


def crawl_website(base_url: str, max_pages: int = 10) -> List[Dict[str, str]]:
    """Crawl pages under base_url, returning list of {url, title, text}."""
    parsed_base = urlparse(base_url)
    base_domain = parsed_base.netloc
    base_path = parsed_base.path.rstrip("/")

    visited = set()
    queue = [base_url]
    pages = []

    while queue and len(pages) < max_pages:
        url = queue.pop(0)
        if url in visited:
            continue
        visited.add(url)

        try:
            text, links = fetch_page(url)
        except Exception as e:
            print(f"  Skip {url}: {e}")
            continue

        # Extract title from first line
        lines = [l for l in text.split("\n") if l.strip()]
        title = lines[0][:100] if lines else url

        pages.append({"url": url, "title": title, "text": text})
        print(f"  Crawled: {url} ({len(text)} chars)")

        # Find same-domain, same-path-prefix links
        for link in links:
            full = urljoin(url, link).split("#")[0].split("?")[0]
            parsed = urlparse(full)
            if (parsed.netloc == base_domain
                    and parsed.path.startswith(base_path)
                    and full not in visited
                    and full.endswith(("/", ".html"))):
                queue.append(full)

        time.sleep(0.3)

    return pages


print("Web scraping functions defined.")


### Fallback: Embedded Sample Content

If the website is unreachable, we use this embedded sample so the notebook still demonstrates the full pipeline.

In [ ]:
SAMPLE_PAGES = [
    {
        "url": "https://docs.python.org/3/faq/general.html",
        "title": "General Python FAQ",
        "text": """General Python FAQ
What is Python? Python is an interpreted, interactive, object-oriented programming language. It incorporates modules, exceptions, dynamic typing, very high level dynamic data types, and classes. Python combines remarkable power with very clear syntax.
Why was Python created? Python was created in the early 1990s by Guido van Rossum at Stichting Mathematisch Centrum in the Netherlands as a successor of a language called ABC.
Is Python a good language for beginning programmers? Yes. Python is a wonderful language for beginning programmers because it is easy to learn and it supports a wide range of applications, from simple text processing to web browsers to games.
What is Python used for? Python is used for web development, data science, machine learning, automation, scripting, scientific computing, and more. Its extensive standard library and third-party packages make it suitable for virtually any programming task.
How does Python compare to other languages? Python emphasizes code readability and simplicity. It runs slower than compiled languages like C or C++ but development speed is much faster. Compared to Java, Python has simpler syntax and dynamic typing.
How stable is Python? Very stable. New stable releases have been coming out roughly every 6 to 18 months since 1991, and this is likely to continue.""",
    },
    {
        "url": "https://docs.python.org/3/faq/programming.html",
        "title": "Programming FAQ",
        "text": """Programming FAQ
How do I share global variables across modules? The canonical way to share information across modules within a single program is to create a special module (often called config or cfg). Just import the config module in all modules of your application and the module becomes available as a global name.
How do I use strings to call functions or methods? There are various techniques. The best is to use a dictionary that maps strings to functions. The primary advantage of this technique is that the strings do not need to match the names of the functions.
How do I create a multidimensional list? You probably tried to make a multidimensional array like this: A = [[None] * 2] * 3. This creates a list containing 3 references to the same list of length two. Modifications to one row will show in all rows. Use a list comprehension instead: A = [[None] * 2 for i in range(3)].
How do I apply a method to a sequence of objects? Use a list comprehension: results = [obj.method() for obj in mylist].
Why does Python use indentation for grouping of statements? Guido van Rossum believes that using indentation for grouping is extremely elegant and contributes a lot to the clarity of the average Python program. Most people learn to love this feature after a while.
How do I convert a number to a string? To convert an integer or float to a string, use the built-in str() function. For hexadecimal or octal, use hex() or oct().
How do I modify a string in place? You cannot, because strings are immutable in Python. If you need a mutable sequence of characters, use a list of characters and join them when done: chars = list(s); chars[0] = 'H'; s = ''.join(chars).
How do I convert between tuples and lists? Use the tuple() and list() built-in functions.""",
    },
    {
        "url": "https://docs.python.org/3/faq/design.html",
        "title": "Design and History FAQ",
        "text": """Design and History FAQ
Why does Python use indentation for grouping? Python uses indentation to indicate block structure. This makes the code easier to read and encourages consistent formatting. The amount of indentation is not fixed, but it must be consistent within a block.
Why is there no switch or case statement in Python? Python did not have a switch statement for a long time. In Python 3.10, structural pattern matching was introduced using the match/case syntax, which is more powerful than a traditional switch statement.
Why must self be explicit in method definitions? Guido van Rossum has written about this choice. By making self explicit, there is no ambiguity about what is an instance variable vs a local variable. It also makes it straightforward to call methods of base classes.
Why can not I use an assignment in an expression? Starting in Python 3.8, you can use the walrus operator := for assignment expressions. Before that, assignment was a statement only. The walrus operator allows assignment within expressions like while (chunk := f.read(8192)).
Why are colons required for if, while, def, class statements? The colon is required mainly to enhance readability. It helps the line stand out and makes the block structure clear.
How fast is Python? Python is not the fastest language. For CPU-intensive work, you can use optimized libraries like NumPy, write extensions in C, or use PyPy. For most applications, Python is fast enough, and developer productivity matters more than raw speed.
Why are there separate tuple and list types? Tuples are immutable and hashable, so they can be used as dictionary keys. Lists are mutable. Tuples are often used for heterogeneous data (like a record), while lists are for homogeneous sequences.""",
    },
    {
        "url": "https://docs.python.org/3/faq/library.html",
        "title": "Library and Extension FAQ",
        "text": """Library and Extension FAQ
How do I find a module or application to perform task X? Check the Python Package Index (PyPI) at https://pypi.org. The standard library also ships with many useful modules. The documentation at docs.python.org lists all standard library modules.
How do I make a Python script executable on Unix? You need to add a shebang line at the top of the script: #!/usr/bin/env python3. Then make the file executable with chmod +x script.py.
Is there a curses or termios package for Python? The standard Python source distribution includes a curses module for Unix systems. The Windows version of Python does not include the curses module.
Is there an equivalent of C's onexit() in Python? The atexit module provides a register function for this purpose.
How do I send mail from a Python script? Use the smtplib module for sending email. The email package helps construct email messages.
How do I generate random numbers in Python? The standard module random implements a random number generator. Use random.random() for a float between 0 and 1, random.randint(a, b) for integers, or random.choice() to pick from a sequence.""",
    },
    {
        "url": "https://docs.python.org/3/faq/extending.html",
        "title": "Extending and Embedding FAQ",
        "text": """Extending and Embedding FAQ
Can I create my own functions in C and call them from Python? Yes, you can create C extension modules. The Python/C API provides functions and macros for creating extension modules. You can also use tools like SWIG, Cython, or cffi.
Can I create my own functions in C++ and call them from Python? Yes, use pybind11 or SWIG. Pybind11 provides a modern C++ interface for creating Python bindings with minimal boilerplate.
How can I execute arbitrary Python statements from C? The highest-level function to do this is PyRun_SimpleString(). It executes a string of Python code in the __main__ module's namespace.
How do I debug an extension? You can use gdb on Unix or Visual Studio on Windows. Compile your extension with debug symbols (-g flag) and attach a debugger to the Python process.
Is there a tool to help find bugs or perform static analysis? Yes, pylint and flake8 are popular tools. Mypy can check type annotations. Bandit checks for security issues.""",
    },
]


print(f"Embedded fallback: {len(SAMPLE_PAGES)} pages, "
      f"{sum(len(p['text']) for p in SAMPLE_PAGES)} total chars")


In [ ]:
# Crawl the live website or use embedded fallback
if HAS_NETWORK:
    print("Crawling live website...")
    pages = crawl_website("https://docs.python.org/3/faq/", max_pages=8)
    if len(pages) < 2:
        print("Crawl returned too few pages, using embedded fallback.")
        pages = SAMPLE_PAGES
else:
    print("Network unavailable, using embedded sample content.")
    pages = SAMPLE_PAGES

print(f"\nTotal pages: {len(pages)}")
for p in pages:
    print(f"  {p['title'][:60]:60s}  {len(p['text']):>6} chars  {p['url']}")


## 9 — Data Inspection

Let's understand the corpus before building the retrieval index.

In [ ]:
# Corpus statistics
total_words = sum(len(p["text"].split()) for p in pages)
total_chars = sum(len(p["text"]) for p in pages)
avg_words = total_words // max(len(pages), 1)

print(f"Pages:          {len(pages)}")
print(f"Total words:    {total_words:,}")
print(f"Total chars:    {total_chars:,}")
print(f"Avg words/page: {avg_words:,}")

# Show a snippet from each page
print("\n--- Page Previews ---")
for p in pages:
    preview = p["text"][:200].replace("\n", " ")
    print(f"\n[{p['title'][:50]}]")
    print(f"  {preview}...")


## 10 — Preprocessing (Text Chunking)

### Why chunk?

Retrieval works best on **focused text segments**, not entire pages:

| Approach | Problem |
|----------|---------|
| Full pages as documents | Too long — dilutes relevance signal |
| Single sentences | Too short — loses context |
| **Overlapping chunks (~300 words)** | **Good balance of focus and context** |

### Chunk parameters

- **chunk_size = 300 words** — large enough for a complete FAQ answer
- **overlap = 50 words** — ensures we don't split an answer across chunks and lose it

In [ ]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """Split text into overlapping word-based chunks."""
    words = text.split()
    if len(words) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


def build_chunks(pages: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Chunk all pages and track source metadata."""
    all_chunks = []
    for page in pages:
        text_chunks = chunk_text(page["text"])
        for i, chunk in enumerate(text_chunks):
            all_chunks.append({
                "text": chunk,
                "source_url": page["url"],
                "source_title": page["title"],
                "chunk_index": i,
            })
    return all_chunks


chunks = build_chunks(pages)
print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c['text'].split()) for c in chunks) // max(len(chunks), 1)} words")
print(f"\nChunk distribution per page:")
for page in pages:
    n = sum(1 for c in chunks if c["source_url"] == page["url"])
    print(f"  {page['title'][:50]:50s} -> {n} chunks")


## 11 — Baseline Approach (Keyword Search)

Before building TF-IDF retrieval, let's establish a **naive keyword search** baseline:
- Split the query into words
- Count how many query words appear in each chunk
- Rank by count

This is the simplest possible retrieval — it will miss synonyms, context, and semantic meaning.

In [ ]:
def keyword_search(query: str, chunks: List[Dict[str, str]], top_k: int = 3) -> List[Dict[str, Any]]:
    """Naive keyword matching — count query word occurrences per chunk."""
    query_words = set(query.lower().split())
    results = []
    for chunk in chunks:
        chunk_lower = chunk["text"].lower()
        score = sum(1 for w in query_words if w in chunk_lower)
        results.append({**chunk, "score": score})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


# Test the baseline
test_query = "How does Python handle memory management?"
baseline_results = keyword_search(test_query, chunks)
print(f"Query: {test_query}")
print(f"\nBaseline results (keyword matching):")
for i, r in enumerate(baseline_results):
    print(f"  [{i+1}] Score: {r['score']} | Source: {r['source_title'][:40]}")
    print(f"      {r['text'][:120]}...")


## 12 — Main Workflow (TF-IDF Retrieval + Answer Generation)

### TF-IDF Retrieval

**TF-IDF** (Term Frequency–Inverse Document Frequency) is a sparse vector method:

| Term | Meaning |
|------|---------|
| TF (Term Frequency) | How often a word appears in a chunk |
| IDF (Inverse Document Frequency) | How rare a word is across all chunks |
| TF-IDF score | TF × IDF — high for words that are frequent here but rare elsewhere |

We vectorize all chunks + the query, then find chunks with highest cosine similarity to the query.

**Why TF-IDF over dense embeddings here?**
- No external model/API needed
- Fast — works on any machine
- Good enough for FAQ-style content where queries and answers share vocabulary
- Dense embeddings (Sentence Transformers, BGE-M3) would be the production upgrade

In [ ]:
class TFIDFRetriever:
    """TF-IDF based document retriever using scikit-learn."""

    def __init__(self, chunks: List[Dict[str, str]]):
        self.chunks = chunks
        self.texts = [c["text"] for c in chunks]
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words="english",
            ngram_range=(1, 2),  # unigrams + bigrams
            sublinear_tf=True,   # apply log normalization
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        print(f"TF-IDF index built: {self.tfidf_matrix.shape[0]} chunks, "
              f"{self.tfidf_matrix.shape[1]} features")

    def search(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Find top-k most relevant chunks for a query."""
        query_vec = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                **self.chunks[idx],
                "score": float(similarities[idx]),
            })
        return results


retriever = TFIDFRetriever(chunks)


### Answer Generation

Two modes:
1. **LLM mode** (Ollama available): Send retrieved chunks + question to the LLM for a synthesized answer
2. **Extractive mode** (fallback): Return the most relevant chunk text directly as the answer

Both modes always show which source pages the answer came from (grounding).

In [ ]:
def call_ollama(prompt: str) -> str:
    """Call Ollama and return the text response."""
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
    }
    req = urllib.request.Request(
        f"{OLLAMA_HOST}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return data.get("response", "").strip()


ANSWER_PROMPT = """You are a helpful FAQ assistant. Answer the user's question based ONLY on the context below.
If the context does not contain enough information, say "I don't have enough information to answer that."
Keep your answer concise (2-4 sentences). Cite which source page the information came from.

Context:
{context}

Question: {question}

Answer:"""


def extractive_answer(results: List[Dict[str, Any]], query: str) -> str:
    """Generate an answer by extracting the most relevant sentences from retrieved chunks."""
    if not results or results[0]["score"] < 0.05:
        return "I don't have enough information to answer that question."

    # Find sentences in top chunk that best match query words
    query_words = set(query.lower().split())
    top_text = results[0]["text"]
    sentences = re.split(r"(?<=[.!?])\s+", top_text)

    scored = []
    for sent in sentences:
        overlap = sum(1 for w in query_words if w in sent.lower())
        scored.append((overlap, sent))
    scored.sort(key=lambda x: x[0], reverse=True)

    # Take top 2-3 most relevant sentences
    answer_sents = [s for _, s in scored[:3] if _]
    if not answer_sents:
        answer_sents = [scored[0][1]] if scored else ["No relevant content found."]

    source = results[0].get("source_title", "Unknown")
    return " ".join(answer_sents) + f"\n\n(Source: {source})"


def ask_question(query: str, retriever: TFIDFRetriever, top_k: int = 3) -> Dict[str, Any]:
    """Full RAG pipeline: retrieve + generate answer."""
    results = retriever.search(query, top_k=top_k)

    if USE_OLLAMA:
        context = "\n\n".join(
            f"[Source: {r['source_title']}]\n{r['text']}"
            for r in results
        )
        answer = call_ollama(ANSWER_PROMPT.format(context=context, question=query))
        method = "LLM (Ollama)"
    else:
        answer = extractive_answer(results, query)
        method = "Extractive (rule-based)"

    return {
        "question": query,
        "answer": answer,
        "method": method,
        "sources": [{"title": r["source_title"], "url": r["source_url"],
                      "score": round(r["score"], 4)} for r in results],
    }


print("RAG pipeline ready.")


## 13 — Pipeline Execution

Let's ask a series of questions to test the FAQ bot.

In [ ]:
TEST_QUESTIONS = [
    "What is Python used for?",
    "How do I convert a number to a string?",
    "Why does Python use indentation?",
    "How do I generate random numbers in Python?",
    "Can I call C functions from Python?",
    "What is the walrus operator?",
]


def display_result(result: Dict[str, Any]) -> None:
    """Pretty-print a Q&A result."""
    print("=" * 70)
    print(f"Q: {result['question']}")
    print(f"\nA: {result['answer']}")
    print(f"\nMethod: {result['method']}")
    print(f"Sources:")
    for s in result["sources"]:
        print(f"  - {s['title'][:50]} (score: {s['score']})")
    print()


all_results = []
for q in TEST_QUESTIONS:
    result = ask_question(q, retriever)
    display_result(result)
    all_results.append(result)


## 14 — Evaluation and Interpretation

### Retrieval Quality

We evaluate **retrieval precision**: does the top result come from the correct FAQ page?

For our known questions, we define expected source pages and check if retrieval finds them.

In [ ]:
# Expected source pages for our test questions
EXPECTED_SOURCES = {
    "What is Python used for?": "general",
    "How do I convert a number to a string?": "programming",
    "Why does Python use indentation?": "design",
    "How do I generate random numbers in Python?": "library",
    "Can I call C functions from Python?": "extending",
    "What is the walrus operator?": "design",
}


def evaluate_retrieval(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Evaluate retrieval quality against expected sources."""
    correct = 0
    total = 0

    print("Retrieval evaluation:")
    print("-" * 70)
    for r in results:
        q = r["question"]
        expected = EXPECTED_SOURCES.get(q, "").lower()
        if not expected:
            continue

        total += 1
        top_source = r["sources"][0]["title"].lower() if r["sources"] else ""
        hit = expected in top_source
        if hit:
            correct += 1

        status = "HIT" if hit else "MISS"
        print(f"  [{status}] {q[:50]}")
        print(f"         Expected: *{expected}* | Got: {top_source[:50]}")

    precision = correct / max(total, 1)
    print(f"\nRetrieval Precision@1: {correct}/{total} = {precision:.1%}")
    return {"precision_at_1": precision, "correct": correct, "total": total}


eval_results = evaluate_retrieval(all_results)


In [ ]:
# Compare keyword baseline vs TF-IDF retrieval
print("Baseline (keyword) vs TF-IDF comparison:")
print("-" * 70)

for q in TEST_QUESTIONS[:4]:
    kw_results = keyword_search(q, chunks, top_k=1)
    tfidf_results = retriever.search(q, top_k=1)

    kw_source = kw_results[0]["source_title"][:30] if kw_results else "None"
    kw_score = kw_results[0]["score"] if kw_results else 0

    tf_source = tfidf_results[0]["source_title"][:30] if tfidf_results else "None"
    tf_score = round(tfidf_results[0]["score"], 4) if tfidf_results else 0

    print(f"  Q: {q[:50]}")
    print(f"    Keyword: {kw_source} (score={kw_score})")
    print(f"    TF-IDF:  {tf_source} (score={tf_score})")
    print()


## 15 — Error Analysis / Limitations

**Retrieval limitations:**
- TF-IDF is vocabulary-dependent — "How do I make Python faster?" may not match chunks about "performance" or "speed" well
- No semantic understanding — can't match "iterate over items" to "loop through elements"
- Short queries with common words produce low-discrimination scores

**Crawling limitations:**
- JavaScript-rendered pages won't be captured (need Playwright/Selenium)
- Pages behind authentication are inaccessible
- Rate limiting / robots.txt may block crawling
- Content behind tabs, accordions, or dynamic widgets is invisible to `HTMLParser`

**Answer generation limitations:**
- Extractive mode returns raw text — no synthesis or rephrasing
- LLM mode may occasionally add information not in the context (hallucination)
- Neither mode handles multi-hop questions well ("Which FAQ page has more questions?")

**What a production system would fix:**
- Dense embeddings (Sentence Transformers) for semantic retrieval
- Hybrid search (sparse + dense) for best of both worlds
- Reranking with a cross-encoder for precision
- Citation verification to catch hallucinated content

## 16 — How to Improve It

### Production Improvement Ideas

1. **Dense embeddings**: Replace TF-IDF with Sentence Transformers or BGE-M3 for semantic matching
2. **Hybrid search**: Combine TF-IDF (exact match) + dense embeddings (semantic) with reciprocal rank fusion
3. **Cross-encoder reranking**: Re-score top-k results with a cross-encoder model for higher precision
4. **Incremental updates**: Re-crawl changed pages only, update index incrementally
5. **Conversation memory**: Track multi-turn conversations so "tell me more about that" works
6. **Source highlighting**: Show which exact sentence in the chunk the answer came from
7. **Scheduled crawling**: Auto-refresh the index daily/weekly to catch content changes
8. **Multi-site support**: Crawl multiple documentation sites into one unified FAQ index

## Common Mistakes

1. **Not chunking at all**
   Feeding entire pages to retrieval dilutes the relevance signal. A 5000-word page that mentions "memory" once will rank poorly even if it's the right page. Chunk it.

2. **Zero overlap between chunks**
   An answer that starts at word 295 and ends at word 310 gets split across two chunks and is lost in both. Always use overlap.

3. **Ignoring retrieval quality**
   Most RAG failures are retrieval failures, not generation failures. If the right chunk isn't retrieved, no LLM can give the right answer. Always evaluate retrieval separately.

4. **No fallback when LLM is down**
   Production systems need a graceful degradation path. Extractive answers are better than error pages.

5. **Crawling too aggressively**
   No delay between requests, no `max_pages` limit, no `robots.txt` check — this gets your IP blocked.

6. **Hardcoding the website**
   Build the pipeline to accept any URL, not just the one you tested with. Otherwise it's a demo, not a tool.

## 17 — Practice Exercises

### Mini Challenge

**Challenge 1:** Point the crawler at a different website (e.g., `https://flask.palletsprojects.com/en/stable/quickstart/`) and build a FAQ bot for Flask documentation. Does TF-IDF retrieval still work well?

**Challenge 2:** Implement a simple **query expansion** — before searching, add synonyms to the query. For example, "fast" → "fast speed performance quick". Does it improve retrieval?

**Challenge 3:** Add a **confidence threshold** — if the top retrieval score is below 0.1, return "I'm not sure, here are the closest topics I found:" instead of a direct answer.

### Try This Next
- Replace TF-IDF with `sentence-transformers` for dense retrieval and compare
- Add a persistent vector store (ChromaDB, FAISS) so you don't re-index on every run
- Build a Streamlit UI with a chat interface for interactive Q&A

## 18 — Final Takeaway / Summary

This notebook built a complete **website FAQ bot** using a minimal RAG pipeline:

1. **Web scraping** — stdlib `HTMLParser` extracts clean text from any website
2. **Chunking** — overlapping word-based chunks preserve context for retrieval
3. **TF-IDF retrieval** — sparse vector search that works offline without GPU or APIs
4. **Grounded answers** — every answer cites which source page it came from
5. **Graceful fallback** — extractive answers when LLM is unavailable

**Key insight:** The quality of a RAG system depends more on retrieval quality than generation quality. If you retrieve the wrong chunks, even the best LLM will give wrong answers. Evaluate retrieval first, then improve generation.

**The production upgrade path:**
```
TF-IDF (this notebook)
  → Sentence Transformers (semantic retrieval)
    → Hybrid search + reranking (production quality)
      → Agentic RAG with multi-step reasoning (advanced)
```